# Error control in recurrent state tracking

Train one-layer models on $S_3$ state tracking with the paper's protocol, then look at
the three diagnostics of Section 4:

1. **Perturbation recovery**: noise injected into the recurrent state is strongly contracted by
   state-dependent models; affine ones only decay it globally or amplify it.
2. **Distinguishability** $q(t) = R(t)/M(t)$: within-class spread over the smallest class
   separation. Affine models cross the readability threshold $1/2$ and then fail.
3. **Where the spread lies**: its part in the symbolic subspace $\mathcal U$ and the rest.

Training the four models takes about four minutes on one GPU.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import torch
import matplotlib.pyplot as plt

from state_tracking.models import LABELS
from state_tracking.train import run_curriculum_training
from state_tracking.analysis import (rollouts, distinguishability, subspace_split,
                                     perturbation_recovery)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Choose the models

Two affine models (Negative Mamba, Token-gated RNN) and two state-dependent ones (tanh RNN,
State-gated RNN), each with its grid-best setting on $S_3$ with one layer in the paper's sweep
and the max-passing length that run reached. On other hardware or PyTorch versions, a run can
end at a different max-passing length.

In [ ]:
PRESETS = {  # mixer: (d_state, learning rate, scheduler, seed, max-passing length in the sweep)
    "neg_mamba":       (64, 1e-3, "fixed",  42,  100),
    "token_gated_rnn": (32, 1e-4, "cosine", 42,  500),
    "tanh_rnn":        (32, 1e-4, "fixed",  42, 1000),
    "state_gated_rnn": (32, 1e-4, "fixed",  42, 1000),
}
MODELS = list(PRESETS)
TASK = "S3"

In [ ]:
trained = {}
for mixer in MODELS:
    d_state, lr, sched, seed, mp_sweep = PRESETS[mixer]
    r = run_curriculum_training(TASK, mixer, n_layers=1, d_state=d_state, lr=lr,
                                scheduler_name=sched, seed=seed, device=device, verbose=False)
    trained[mixer] = r
    print(f"{LABELS[mixer]:16s} max-passing length {r['max_passing']:4d} "
          f"(sweep: {mp_sweep})   {r['total_epochs']} epochs, {r['elapsed']:.0f}s")

## 1. Perturbation recovery

Run a clean and a perturbed copy of the first layer's recurrence on the same inputs. At step
$t_0 = 20$ add Gaussian noise ($\sigma = 10^{-2}$) to the perturbed state, then track
$\|e_t\| / \|e_{t_0}\|$ over 200 sequences (median and interquartile range).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
for mixer in MODELS:
    ratio = perturbation_recovery(trained[mixer]["model"], TASK, device)
    med = np.median(ratio, axis=0)
    q25, q75 = np.percentile(ratio, [25, 75], axis=0)
    steps = np.arange(ratio.shape[1])
    ax.plot(steps, med, label=LABELS[mixer])
    ax.fill_between(steps, q25, q75, alpha=0.2)
ax.axhline(1.0, color="gray", ls="--", lw=0.8)
ax.set_yscale("log")
ax.set_xlabel("steps after injection")
ax.set_ylabel(r"$\|e_t\| / \|e_{t_0}\|$")
ax.legend(frameon=False)
plt.show()

## 2. Distinguishability over long rollouts

For 200 rollouts of length 1000, with $c_g(t)$ the mean state of the rollouts whose symbolic
state is $g$ at step $t$:
$R(t)$ is the mean distance of a state to its class centroid, $M(t)$ the smallest distance
between two centroids, and $q(t) = R(t)/M(t)$. In readout space ($W_{\text{out}} h$, solid),
$q(t) < 1/2$ means that a state at the mean distance $R(t)$ is still closer to its own centroid
than to any other; latent space ($h$) is dashed. The dotted vertical line is the model's
max-passing length.

In [ ]:
ROLLOUT_LEN = 1000
states = {m: rollouts(trained[m]["model"], TASK, n_seqs=200, seq_len=ROLLOUT_LEN,
                      seed=42, device=device) for m in MODELS}

fig, axes = plt.subplots(1, len(MODELS), figsize=(3.2 * len(MODELS), 3), sharey=True)
t = np.arange(1, ROLLOUT_LEN + 1)
for ax, mixer in zip(np.atleast_1d(axes), MODELS):
    h, labels = states[mixer]
    readout = distinguishability(h, labels, model=trained[mixer]["model"])
    latent = distinguishability(h, labels)
    line, = ax.plot(t, readout["q"], label="readout")
    ax.plot(t, latent["q"], ls="--", color=line.get_color(), alpha=0.6, label="latent")
    ax.axhline(0.5, color="gray", ls="--", lw=0.8)
    if trained[mixer]["max_passing"] > 0:
        ax.axvline(trained[mixer]["max_passing"], color="gray", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_ylim(1e-2, None)
    ax.set_title(LABELS[mixer]); ax.set_xlabel("step $t$")
np.atleast_1d(axes)[0].set_ylabel("$q(t)$")
np.atleast_1d(axes)[0].legend(frameon=False)
plt.tight_layout(); plt.show()

## 3. Where the spread lies

Split each state's deviation from its class centroid into the part in the symbolic subspace
$\mathcal U(t)$, spanned by the centered centroids, and the orthogonal rest. Both are root mean
squares over rollouts, in units of the smallest centroid distance (latent space).
Once an affine model preserves the state representations, it cannot contract errors along
$\mathcal U$ (Theorem 1).

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(3.2 * len(MODELS), 3), sharey=True)
for ax, mixer in zip(np.atleast_1d(axes), MODELS):
    split = subspace_split(*states[mixer])
    line, = ax.plot(t, split["q_U"], label=r"in $\mathcal{U}$")
    ax.plot(t, split["q_perp"], color="gray", label=r"orthogonal to $\mathcal{U}$")
    if trained[mixer]["max_passing"] > 0:
        ax.axvline(trained[mixer]["max_passing"], color="gray", ls=":", lw=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(LABELS[mixer]); ax.set_xlabel("step $t$")
np.atleast_1d(axes)[0].set_ylabel("spread / centroid separation")
np.atleast_1d(axes)[0].legend(frameon=False)
plt.tight_layout(); plt.show()